# BombermanAI Round Robin Benchmark

This Kaggle notebook is used to:
- clone the repo from GitHub
- install dependencies
- run the same seed set for every agent
- compare two environments: normal and fog of war
- export detailed tables for the report
- record decision time per step for speed comparison

Notes:
- Replace `REPO_URL` with your real GitHub link.
- The benchmark here is a round-robin style comparison over the same seed range so the results are fair and reproducible.


In [ ]:
# 1) Clone repository
import os
import sys
import math
import json
import statistics
from pathlib import Path

REPO_URL = "https://github.com/<your-user-or-org>/Boomber-AI-game.git"
REPO_DIR = "Boomber-AI-game"

if not Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# 3) Import benchmark helpers and agent factories
import pandas as pd

from benchmarks.benchmark import run_game
from agents.random_agent import RandomAgent
from agents.bfs_agent import BFSAgent
from agents.dfs_agent import DFSAgent
from agents.greedy_agent import GreedyAgent
from agents.astar_agent import AStarAgent
from agents.hill_climbing_agent import HillClimbingAgent
from agents.simulated_annealing_agent import SimulatedAnnealingAgent
from agents.backtracking_agent import BacktrackingAgent
from agents.min_conflicts_agent import MinConflictsAgent
from agents.online_search_agent import OnlineSearchAgent
from agents.and_or_search_agent import AndOrSearchAgent
from agents.minimax_agent import MinimaxAgent
from agents.expectimax_agent import ExpectimaxAgent


def make_random():
    return RandomAgent(seed=42)

def make_bfs():
    return BFSAgent()

def make_dfs():
    return DFSAgent()

def make_greedy():
    return GreedyAgent()

def make_astar():
    return AStarAgent()

def make_hill_climbing():
    return HillClimbingAgent()

def make_sim_annealing():
    return SimulatedAnnealingAgent()

def make_backtracking():
    return BacktrackingAgent()

def make_min_conflicts():
    return MinConflictsAgent()

def make_online_search():
    return OnlineSearchAgent()

def make_and_or_search():
    return AndOrSearchAgent()

def make_minimax():
    return MinimaxAgent(depth=2)

def make_expectimax():
    return ExpectimaxAgent(depth=2)


AGENTS = [
    ("Random", make_random),
    ("BFS", make_bfs),
    ("DFS", make_dfs),
    ("Greedy", make_greedy),
    ("A*", make_astar),
    ("HillClimbing", make_hill_climbing),
    ("SimulatedAnnealing", make_sim_annealing),
    ("Backtracking", make_backtracking),
    ("MinConflicts", make_min_conflicts),
    ("OnlineSearch", make_online_search),
    ("AndOrSearch", make_and_or_search),
    ("Minimax", make_minimax),
    ("Expectimax", make_expectimax),
]

print("Agents loaded:", [name for name, _ in AGENTS])


In [ ]:
# 3) Import benchmark helpers and agent factories
import pandas as pd

from benchmarks.benchmark import run_game
from agents.random_agent import RandomAgent
from agents.bfs_agent import BFSAgent
from agents.dfs_agent import DFSAgent
from agents.greedy_agent import GreedyAgent
from agents.astar_agent import AStarAgent
from agents.hill_climbing_agent import HillClimbingAgent
from agents.simulated_annealing_agent import SimulatedAnnealingAgent
from agents.backtracking_agent import BacktrackingAgent
from agents.min_conflicts_agent import MinConflictsAgent
from agents.online_search_agent import OnlineSearchAgent
from agents.and_or_search_agent import AndOrSearchAgent
from agents.minimax_agent import MinimaxAgent
from agents.expectimax_agent import ExpectimaxAgent


AGENTS = [
    ("Random", lambda: RandomAgent(seed=42)),
    ("BFS", lambda: BFSAgent()),
    ("DFS", lambda: DFSAgent()),
    ("Greedy", lambda: GreedyAgent()),
    ("A*", lambda: AStarAgent()),
    ("HillClimbing", lambda: HillClimbingAgent()),
    ("SimulatedAnnealing", lambda: SimulatedAnnealingAgent()),
    ("Backtracking", lambda: BacktrackingAgent()),
    ("MinConflicts", lambda: MinConflictsAgent()),
    ("OnlineSearch", lambda: OnlineSearchAgent()),
    ("AndOrSearch", lambda: AndOrSearchAgent()),
    ("Minimax", lambda: MinimaxAgent(depth=2)),
    ("Expectimax", lambda: ExpectimaxAgent(depth=2)),
]

print("Agents loaded:", [name for name, _ in AGENTS])

In [ ]:
# 4) Tournament mode: 4-agent round robin with ELO (multi-core)
import os
import sys
from pathlib import Path

# Make backend/app importable on Kaggle
PROJECT_ROOT = Path.cwd()
BACKEND_DIR = PROJECT_ROOT / "Boomber-AI-game" / "backend"
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))
import math
from concurrent.futures import ProcessPoolExecutor

from app.engine import SimulationEngine
from app.metrics_tracker import MetricsTracker


def make_tournament_roster():
    return AGENTS


def _build_actions_from_state(agent_name, agent_factory, state_dict, cache):
    agent = cache.get(agent_name)
    if agent is None:
        agent = agent_factory()
        cache[agent_name] = agent
    action_str = agent.choose_action(state_dict)
    return action_str


def _run_tournament_match(args):
    match_id, agents_pair, seed = args
    roster = dict(agents_pair)
    order = list(roster.keys())
    engine = SimulationEngine(seed=seed)
    tracker = MetricsTracker(agent_ids=order)
    engine.tracker = tracker

    # Spawn the 4 bots in the four corners.
    spawn_points = [(1, 1), (13, 1), (1, 11), (13, 11)]
    for pid, (x, y) in zip(order, spawn_points):
        engine.add_agent(pid, x, y, lives=1)

    bot_instances = {pid: roster[pid]() for pid in order}
    done = False
    while not done:
        actions = {}
        for pid in order:
            agent_state = engine.get_state()
            # Rebuild a player-centric state for the current agent.
            current = engine.state.agents[pid]
            walls = []
            bricks = []
            items = {}
            for y in range(engine.state.height):
                for x in range(engine.state.width):
                    tile = engine.state.grid[y][x]
                    if tile == 1:
                        walls.append((x, y))
                    elif tile == 2:
                        bricks.append((x, y))
                    elif tile == 5:
                        items[(x, y)] = 1
                    elif tile == 6:
                        items[(x, y)] = 2

            enemy_positions = [(a.x, a.y) for aid, a in engine.state.agents.items() if aid != pid and a.is_alive]
            bomb_positions = [(b.x, b.y, max(1, b.timer - 3)) for b in engine.state.bombs]
            explosions = list(engine.state.explosions)
            state_for_agent = {
                "self_position": (current.x, current.y),
                "enemy_positions": enemy_positions,
                "bomb_positions": bomb_positions,
                "explosions": explosions,
                "walls": walls,
                "bricks": bricks,
                "items": items,
                "ammo": current.bombs_left,
                "blast_radius": current.bomb_range,
                "width": engine.state.width,
                "height": engine.state.height,
                "legal_actions": ["WAIT", "LEFT", "RIGHT", "UP", "DOWN", "BOMB"],
            }

            t0 = __import__("time").perf_counter()
            action_str = bot_instances[pid].choose_action(state_for_agent)
            t1 = __import__("time").perf_counter()
            if tracker:
                tracker.record_step_latency(pid, (t1 - t0) * 1000.0)

            action_map = {"WAIT": 0, "STOP": 0, "LEFT": 1, "RIGHT": 2, "UP": 3, "DOWN": 4, "BOMB": 5}
            actions[pid] = action_map.get(action_str, 0)

        done = not engine.step(actions)

    alive_agents = [aid for aid, a in engine.state.agents.items() if a.is_alive]
    winner_id = alive_agents[0] if len(alive_agents) == 1 else None
    stats = tracker.get_final_stats(engine.state.step_count, alive_agents)
    # Rank 1 => winner, but in ties the tracker already orders by survival.
    return {
        "match_id": match_id,
        "seed": seed,
        "steps": engine.state.step_count,
        "winner_id": winner_id,
        "stats": stats,
    }


def _expected_score(rank_a, rank_b, scale=400.0):
    return 1.0 / (1.0 + 10 ** ((rank_b - rank_a) / scale))


def _update_elo(ratings, match_stats, k=24):
    # Lower rank is better. For 4-player matches, use pairwise implied results.
    ordered = sorted(match_stats, key=lambda r: (r["rank"], -r["survival_steps"]))
    for i, a in enumerate(ordered):
        aid = a["agent_id"]
        ratings.setdefault(aid, 1500.0)
        score = 0.0
        expected = 0.0
        for j, b in enumerate(ordered):
            if i == j:
                continue
            bid = b["agent_id"]
            ratings.setdefault(bid, 1500.0)
            if a["rank"] < b["rank"]:
                actual = 1.0
            elif a["rank"] > b["rank"]:
                actual = 0.0
            else:
                actual = 0.5
            score += actual
            expected += 1.0 / (1.0 + 10 ** ((ratings[bid] - ratings[aid]) / 400.0))
        n = max(1, len(ordered) - 1)
        ratings[aid] = ratings[aid] + k * ((score / n) - (expected / n))


def run_tournament_round_robin(*, num_games=1000, start_seed=42, workers=None):
    workers = workers or max(1, os.cpu_count() or 1)
    roster = make_tournament_roster()
    agent_names = [name for name, _ in roster]
    matchups = []
    seed_cursor = start_seed
    # All 4-agent combinations. Each match is a real 4-player battle.
    from itertools import combinations
    combs = list(combinations(roster, 4))
    total_matches = num_games * len(combs)
    for game_idx in range(num_games):
        for combo in combs:
            matchups.append((f"g{game_idx}", combo, seed_cursor))
            seed_cursor += 1

    print(f"Tournament mode: {len(agent_names)} agents")
    print(f"Total matches: {total_matches}")
    print(f"Using {workers} worker processes (os.cpu_count={os.cpu_count()})")

    all_match_rows = []
    ratings = {name: 1500.0 for name in agent_names}
    completed = 0
    with ProcessPoolExecutor(max_workers=workers) as executor:
        for result in executor.map(_run_tournament_match, matchups, chunksize=max(1, math.ceil(len(matchups) / (workers * 4)))):
            completed += 1
            print(f"Completed {completed}/{total_matches} matches")
            stats = result["stats"]
            _update_elo(ratings, stats)
            for row in stats:
                all_match_rows.append({
                    "match_id": result["match_id"],
                    "seed": result["seed"],
                    "steps": result["steps"],
                    "winner_id": result["winner_id"],
                    **row,
                    "elo_after_match": round(ratings[row["agent_id"]], 2),
                })

    results_df = pd.DataFrame(all_match_rows)
    summary_df = results_df.groupby("agent_id", as_index=False).agg(
        matches_played=("match_id", "count"),
        wins=("rank", lambda s: int((s == 1).sum())),
        average_rank=("rank", "mean"),
        average_survival_steps=("survival_steps", "mean"),
        average_latency_ms=("avg_latency_ms", "mean"),
        kills=("kills", "sum"),
        suicides=("suicides", "sum"),
        final_elo=("elo_after_match", "last"),
    )
    summary_df["win_rate"] = summary_df["wins"] / summary_df["matches_played"]
    summary_df = summary_df.sort_values(["final_elo", "win_rate", "average_rank"], ascending=[False, False, True]).reset_index(drop=True)
    return results_df, summary_df, ratings


NUM_GAMES = 1000
START_SEED = 42
WORKERS = max(1, os.cpu_count() or 1)

# For smoke test only:
# NUM_GAMES = 20
# WORKERS = 2

results_df, summary_df, elo_ratings = run_tournament_round_robin(
    num_games=NUM_GAMES,
    start_seed=START_SEED,
    workers=WORKERS,
)

summary_df


In [ ]:
# 5) Tournament summary tables
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

summary_df = summary_df.sort_values(["final_elo", "win_rate", "average_rank"], ascending=[False, False, True]).reset_index(drop=True)
summary_df


In [ ]:
# 6) Tournament diagnostics
results_df.head()


In [ ]:
# 7) Export artifacts for report writing
output_dir = Path("benchmark_outputs")
output_dir.mkdir(exist_ok=True)

results_path = output_dir / "tournament_round_robin_raw_results.csv"
summary_path = output_dir / "tournament_round_robin_summary.csv"
elo_path = output_dir / "tournament_elo.csv"

results_df.to_csv(results_path, index=False)
summary_df.to_csv(summary_path, index=False)
pd.DataFrame(sorted(elo_ratings.items(), key=lambda x: x[1], reverse=True), columns=["agent_id", "elo"]).to_csv(elo_path, index=False)

print("Saved:")
print(results_path)
print(summary_path)
print(elo_path)


## Reporting tips

In the experimental section, you can report:
- tournament mode with 4 agents per match
- total matches and completed matches
- win rate, average rank, average survival steps
- ELO ranking after the round robin tournament
- average latency per agent
- use the same seeds to keep the comparison fair

If needed, you can also add:
- head-to-head win matrix
- top-k and bottom-k seeds for each agent
- stability comparison across tournament rounds
